In [ ]:
# -*- coding: utf-8 -*-
"""scbert_finetune_real_data.ipynb

# scBERT Pretrained Fine-Tuning Benchmark — Phase 8

**What this notebook tests:**
Fine-tuning the *pretrained* scBERT transformer on real datasets (80/20 train/test split).
Instead of simulated scenarios with replicates, this notebook iterates over all independent
`.h5ad` samples in the `h5ad_real` directory.

**GPU:** L4 (22.5 GB VRAM).
**Crash safety:** Each run saves immediately to Drive. Re-running skips completed runs.
"""

from google.colab import drive
drive.mount('/content/drive')

!pip install -q performer-pytorch==1.1.4 einops scanpy anndata scikit-learn

import os
if not os.path.exists('/content/scBERT'):
    !git clone -q https://github.com/TencentAILabHealthcare/scBERT /content/scBERT
    print("scBERT repo cloned")
else:
    print("scBERT repo already present")

In [ ]:
import os

# Path to your uploaded Gene2Vec file on Drive
DRIVE_G2V_PATH = "/content/drive/MyDrive/Thesis FM Computing/weights/gene2vec_16906.npy"

# Create links in both locations scBERT might look
# 1. /content/data/ (standard root relative)
# 2. /content/scBERT/data/ (relative to the repo internal structure)

LINK_LOCATIONS = [
    "/content/data",
    "/content/scBERT/data"
]

if os.path.exists(DRIVE_G2V_PATH):
    for loc in LINK_LOCATIONS:
        os.makedirs(loc, exist_ok=True)
        target = os.path.join(loc, "gene2vec_16906.npy")

        if os.path.exists(target) or os.path.islink(target):
            os.remove(target)

        os.symlink(DRIVE_G2V_PATH, target)
        print(f"✅ Linked to {target}")
else:
    print(f"❌ Error: File not found at {DRIVE_G2V_PATH}. Please check your Drive path.")

In [ ]:
import os
import sys
import numpy as np
import performer_pytorch

# The code showed scBERT looks for '../data/gene2vec_16906.npy' relative to its script location.
# Script: /content/scBERT/performer_pytorch/performer_pytorch.py
# Relative '../data/' resolves to: /content/scBERT/data/

DRIVE_G2V_PATH = '/content/drive/MyDrive/Thesis FM Computing/weights/gene2vec_16906.npy'
REQUIRED_DIR = '/content/scBERT/data'
TARGET_PATH = os.path.join(REQUIRED_DIR, 'gene2vec_16906.npy')

os.makedirs(REQUIRED_DIR, exist_ok=True)

if os.path.exists(DRIVE_G2V_PATH):
    if os.path.exists(TARGET_PATH) or os.path.islink(TARGET_PATH):
        os.remove(TARGET_PATH)

    os.symlink(DRIVE_G2V_PATH, TARGET_PATH)
    print(f"✅ Successfully linked {DRIVE_G2V_PATH} to {TARGET_PATH}")

    # Verification for the relative path scBERT uses
    try:
        test_load = np.load(TARGET_PATH)
        print(f"✅ Verification: File is readable at {TARGET_PATH}. Shape: {test_load.shape}")
    except Exception as e:
        print(f"❌ Verification failed: {e}")
else:
    print(f"❌ Error: Source file not found at {DRIVE_G2V_PATH}")

In [ ]:
# Let's inspect the specific lines in the library responsible for loading the file
import subprocess

file_to_check = "/content/scBERT/performer_pytorch/performer_pytorch.py"

print(f"Searching for 'gene2vec' references in {file_to_check}:\n")

try:
    # Grep for the filename and show context lines
    result = subprocess.check_output(["grep", "-C", "5", "gene2vec", file_to_check], text=True)
    print(result)
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
import glob
import os

# ── Allocator config — reduces fragmentation, set before any CUDA ops ─────
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ── EDIT THESE TO MATCH YOUR DRIVE ────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/Thesis FM Computing"
H5AD_DIR        = f"{DRIVE_ROOT}/h5ad_real"
RESULTS_DIR     = f"{DRIVE_ROOT}/phase8_results"
CKPT_DIR        = f"{DRIVE_ROOT}/phase8_checkpoints/scbert_finetune"

# Path to the pretrained weights
PRETRAINED_CKPT = "/content/drive/MyDrive/Thesis FM Computing/weights/panglao_pretrain.pth"

# Path to the exact 16,906 gene list (one gene symbol per line). If absent, the panel is derived from
# the scBERT reference AnnData (panglao_10000.h5ad) below and cached here.
GENE_VOCAB_PATH = f"{DRIVE_ROOT}/panglao_gene_vocab.txt"
# scBERT reference AnnData; its var_names ARE the canonical 16,906-gene order (aligned to gene2vec_16906.npy).
# Part of the official scBERT data bundle (same one that ships gene2vec_16906.npy / panglao_pretrain.pth).
PANGLAO_REF_H5AD = f"{DRIVE_ROOT}/weights/panglao_10000.h5ad"
# ─────────────────────────────────────────────────────────────────────────

MODEL_NAME = "scBERT_pretrained"
TRAIN_SEED = 42

# Architecture
EMB_DIM      = 200
DEPTH        = 6
NUM_HEADS    = 10
MAX_VALUE    = 6
SEQ_LEN      = 16906

# Fine-tuning hyperparameters
BATCH_SIZE   = 2
ACCUM_STEPS  = 4
MAX_EPOCHS   = 30
LR           = 1e-4
PATIENCE     = 5
WEIGHT_DECAY = 1e-4

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)

sample_paths = sorted(glob.glob(os.path.join(H5AD_DIR, "*.h5ad")))
sample_paths = [p for p in sample_paths if "_remapped" not in p]
SAMPLES = [os.path.basename(p).replace(".h5ad", "") for p in sample_paths]

print(f"Found {len(SAMPLES)} samples.")

In [ ]:
import sys, time, warnings, random, gc, urllib.request
warnings.filterwarnings("ignore")

# Forcefully remove the pip-installed package from cache if it was loaded
for key in list(sys.modules.keys()):
    if key.startswith("performer_pytorch"):
        del sys.modules[key]

# Prioritize the local scBERT repository
sys.path.insert(0, "/content/scBERT")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import issparse

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, f1_score,
                              matthews_corrcoef, accuracy_score)

# Load PerformerLM explicitly from the scBERT custom files
try:
    from performer_pytorch import PerformerLM
    # Quick check to ensure it's the right one
    import inspect
    if 'g2v_position_emb' not in inspect.signature(PerformerLM.__init__).parameters:
        raise ImportError("Loaded the wrong performer_pytorch. Falling back to direct file load.")
    print("performer_pytorch imported OK (scBERT custom version)")
except Exception:
    print("Directly loading from file...")
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "performer_pytorch",
        "/content/scBERT/performer_pytorch/performer_pytorch.py"
    )
    custom_performer = importlib.util.module_from_spec(spec)
    sys.modules["performer_pytorch"] = custom_performer
    spec.loader.exec_module(custom_performer)
    PerformerLM = custom_performer.PerformerLM
    print("performer_pytorch forcefully loaded from scBERT repository")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")

In [ ]:
GENE_LIST_LOCAL = "/tmp/scbert_gene_panel.txt"

def ensure_raw_counts(adata):
    # Raw integer counts are stored in adata.layers['counts']; adata.X is z-scored (has negatives).
    # Prefer the counts layer, then .raw, then fall back to X (guarded).
    if "counts" in adata.layers:
        X, var = adata.layers["counts"], adata.var
    elif adata.raw is not None:
        X, var = adata.raw.X, adata.raw.var
    else:
        X, var = adata.X, adata.var
    if issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    if X.min() < 0:
        raise ValueError(
            "Count matrix has negative values \u2014 layers['counts'] is not raw counts "
            "(adata.X is z-scored). Cannot tokenize."
        )
    return ad.AnnData(
        X   = np.round(X).astype(np.float32),
        obs = adata.obs.copy(),
        var = var.copy(),
        uns = adata.uns.copy()
    )

def _genes_from_panglao_ref(ref_path):
    """Canonical 16,906-gene order = panglao_10000.h5ad.var_names (scBERT's reference panel;
    gene2vec_16906.npy rows align to it). Read names only via backed mode (no X load)."""
    aref = ad.read_h5ad(ref_path, backed="r")
    genes = [str(g) for g in aref.var_names]
    try:
        aref.file.close()
    except Exception:
        pass
    return genes

def get_gene_panel(vocab_path=GENE_VOCAB_PATH, ref_h5ad=PANGLAO_REF_H5AD, n_genes=16906):
    """
    Load the EXACT 16,906-gene panglao panel whose ORDER matches gene2vec_16906.npy and
    panglao_pretrain.pth. The order is load-bearing: panel position i must be the gene the pretrained
    gene2vec/positional embedding row i was trained on. There is deliberately NO arbitrary fallback —
    an HGNC list silently destroys semantic transfer (the old bug: "No genes matched" / kappa=0).
    Source priority: (1) the plain-text vocab file; (2) derive from panglao_10000.h5ad var_names and cache.
    """
    if os.path.exists(vocab_path):
        with open(vocab_path, "r") as f:
            try:
                import json
                data = json.load(f)
                genes = list(data.keys()) if isinstance(data, dict) else data
            except Exception:
                f.seek(0)
                genes = [l.strip() for l in f if l.strip()]
        print(f"Loaded scBERT gene panel from {vocab_path} ({len(genes)} genes).")
    elif os.path.exists(ref_h5ad):
        genes = _genes_from_panglao_ref(ref_h5ad)
        print(f"Derived scBERT gene panel from {ref_h5ad} var_names ({len(genes)} genes).")
        try:
            with open(vocab_path, "w") as f:
                f.write("\n".join(genes))
            print(f"  Cached panel -> {vocab_path} for future runs.")
        except Exception as e:
            print(f"  [WARN] Could not cache panel: {e}")
    else:
        raise FileNotFoundError(
            "scBERT gene panel not found. Provide ONE of:\n"
            f"  (a) a gene list at {vocab_path} (one symbol per line, official 16,906 order), OR\n"
            f"  (b) the scBERT reference {ref_h5ad} (panglao_10000.h5ad; panel = its var_names).\n"
            "Both ship in the official scBERT data bundle (README download links at "
            "https://github.com/TencentAILabHealthcare/scBERT) — the SAME bundle as gene2vec_16906.npy "
            "and panglao_pretrain.pth. Do NOT substitute an arbitrary gene list; order must match gene2vec."
        )
    if len(genes) < n_genes:
        raise ValueError(
            f"Gene panel has only {len(genes)} genes but scBERT requires exactly {n_genes} "
            f"(matching the pretrained gene2vec/weights). Wrong or truncated panel."
        )
    if len(genes) != n_genes:
        print(f"  [WARN] Panel has {len(genes)} entries; using the first {n_genes}. "
              f"Confirm this ordering matches gene2vec_16906.npy.")
    return genes[:n_genes]

GENE_PANEL = get_gene_panel()

In [ ]:
def load_sample(sample_name):
    """Load real data sample."""
    raw_path = os.path.join(H5AD_DIR, f"{sample_name}.h5ad")

    if not os.path.exists(raw_path):
        raise FileNotFoundError(f"Cannot find: {raw_path}")

    adata = sc.read_h5ad(raw_path)
    adata = ensure_raw_counts(adata)

    # Fix: Map alternative ground truth column names to 'cell_type'
    if "cell_type" not in adata.obs.columns:
        if "Ground_Truth_Celltype" in adata.obs.columns:
            adata.obs["cell_type"] = adata.obs["Ground_Truth_Celltype"]
            print(f"  [INFO] Mapped 'Ground_Truth_Celltype' to 'cell_type'")
        else:
            raise KeyError(f"'cell_type' column missing. Available: {list(adata.obs.columns)}")

    n_types = adata.obs["cell_type"].nunique()
    print(f"  Loaded {sample_name}: {adata.shape[0]} cells × {adata.shape[1]} genes | {n_types} cell types")
    return adata

def stratified_split(adata, test_size=0.2):
    """Stratified 80/20 split."""
    #Remove cells with NaN cell_type
    initial_count = len(adata)
    adata = adata[~adata.obs["cell_type"].isna()].copy()
    if len(adata) < initial_count:
        print(f"  [FIX] Removed {initial_count - len(adata)} cells with missing cell types.")

    counts = adata.obs["cell_type"].value_counts()
    singletons = counts[counts < 2].index.tolist()
    if singletons:
        print(f"  [FIX] Removing singleton classes: {singletons}")
        adata = adata[~adata.obs["cell_type"].isin(singletons)].copy()

    labels = adata.obs["cell_type"].values
    idx    = np.arange(len(labels))
    tr_idx, te_idx = train_test_split(
        idx, test_size=test_size, stratify=labels, random_state=TRAIN_SEED
    )
    return adata[tr_idx].copy(), adata[te_idx].copy()

def encode_labels(train_a, test_a):
    all_types = sorted(train_a.obs["cell_type"].unique())
    label2id  = {t: i for i, t in enumerate(all_types)}
    id2label  = {i: t for t, i in label2id.items()}
    y_train   = np.array([label2id[t] for t in train_a.obs["cell_type"]])
    y_test    = np.array([label2id[t] for t in test_a.obs["cell_type"]])
    return label2id, id2label, y_train, y_test

def to_dense(adata):
    X = adata.X
    if issparse(X): X = X.toarray()
    return X.astype(np.float32)

In [ ]:
def tokenise_for_scbert(X_raw, gene_names, gene_panel):
    """
    To use pretrained weights, the output MUST be exactly 16,906 length.
    We map the dataset's real genes to their corresponding index in the scBERT
    reference panel. Missing genes are left as 0 (which behaves correctly as
    a pad/zero-expression token in scBERT).
    """
    n_cells = X_raw.shape[0]
    n_panel = len(gene_panel) # Should be 16906

    # Create mapping from panel gene to its index
    panel_idx = {g: i for i, g in enumerate(gene_panel)}

    # Find overlapping genes
    matched_data_idx = []
    matched_panel_idx = []
    for i, g in enumerate(gene_names):
        if g in panel_idx:
            matched_data_idx.append(i)
            matched_panel_idx.append(panel_idx[g])

    pct_match = 100.0 * len(matched_data_idx) / len(gene_panel)
    if len(matched_data_idx) == 0:
        raise ValueError(
            "No genes matched between data and the scBERT panglao panel. The dataset's gene identifiers "
            "are not symbols present in the panel (e.g. they may be Ensembl IDs, or need NCBI gene-symbol "
            "revision per the scBERT README). Cannot tokenize this sample."
        )
    if pct_match < 5:
        print(f"  [WARNING] Only {len(matched_data_idx)}/{len(gene_panel)} ({pct_match:.1f}%) genes matched "
              f"the panel — scBERT input will be near-empty for this sample.")

    # Library-size normalise → log1p → floor + clip [0, MAX_VALUE]
    # Official scBERT recipe (no per-cell /max rescale; the model was pretrained on this distribution).
    lib   = X_raw.sum(axis=1, keepdims=True)
    lib   = np.where(lib == 0, 1.0, lib)
    X_log = np.log1p((X_raw / lib) * 10_000)
    X_bin = np.clip(np.floor(X_log).astype(np.int8), 0, MAX_VALUE)

    # Initialize the full 16,906 sequence with zeros
    out = np.zeros((n_cells, n_panel), dtype=np.int8)

    # Place matched genes in their exact pretrained coordinate positions
    for d_idx, p_idx in zip(matched_data_idx, matched_panel_idx):
        out[:, p_idx] = X_bin[:, d_idx]

    tensor_mb = out.nbytes / 1024**2
    pct_panel = 100.0 * len(matched_data_idx) / len(gene_panel)
    pct_data  = 100.0 * len(matched_data_idx) / max(len(gene_names), 1)
    print(f"  Matched: {len(matched_data_idx)} genes "
          f"({pct_data:.1f}% of {len(gene_names)} data genes | {pct_panel:.1f}% of {len(gene_panel)} panel) | "
          f"tensor: {n_cells}×{n_panel} = {tensor_mb:.0f} MB")

    return torch.tensor(out.astype(np.int64), dtype=torch.long), n_panel

In [ ]:
class scBERTClassifier(nn.Module):
    def __init__(self, num_classes, seq_len=16906, pretrained_path=None):
        super().__init__()

        # nb_features=266 is required to match the panglao_pretrain.pth checkpoint
        self.backbone = PerformerLM(
            num_tokens       = MAX_VALUE + 1,
            dim              = EMB_DIM,
            depth            = DEPTH,
            heads            = NUM_HEADS,
            max_seq_len      = seq_len,
            local_attn_heads = 0,
            g2v_position_emb = True,
            nb_features      = 266,
        )

        self.head = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=1, kernel_size=(1, EMB_DIM)),
            nn.ReLU(),
            nn.Flatten(start_dim=1),
            nn.Linear(in_features=seq_len, out_features=512, bias=True),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Linear(in_features=512, out_features=100, bias=True),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Linear(in_features=100, out_features=num_classes, bias=True)
        )

        self._random_init()

        if pretrained_path and os.path.exists(pretrained_path):
            self._load_pretrained(pretrained_path)
        else:
            print(f"  [WARNING] Pretrained path '{pretrained_path}' not found!")

        n_params = sum(p.numel() for p in self.parameters())
        print(f"  scBERT: seq_len={seq_len}, {num_classes} classes, {n_params:,} params")

    def _random_init(self):
        def _init(m):
            if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        self.apply(_init)

    def _load_pretrained(self, pretrained_path):
        print(f"  Loading pretrained weights from {pretrained_path}...")
        ckpt = torch.load(pretrained_path, map_location="cpu")
        if 'model_state_dict' in ckpt:
            ckpt = ckpt['model_state_dict']

        new_dict = {}
        for k, v in ckpt.items():
            new_k = k.replace("module.", "").replace("encoder.", "").replace("backbone.", "")
            new_dict[new_k] = v

        # nb_features mismatch fix: weights will now align with nb_features=266
        missing, unexpected = self.backbone.load_state_dict(new_dict, strict=False)
        print(f"  Pretrained backbone loaded. (Missing: {len(missing)}, Unexpected: {len(unexpected)})")

    def forward(self, x):
        emb = self.backbone(x, return_encodings=True)
        emb = emb.unsqueeze(1)
        return self.head(emb)

In [ ]:
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total, n = 0.0, 0
    optimizer.zero_grad()

    for step, (Xb, yb) in enumerate(loader):
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(Xb)
            loss = criterion(logits.float(), yb) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total += loss.item() * ACCUM_STEPS * len(yb)
        n += len(yb)

    return total / n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total, n, preds = 0.0, 0, []
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(Xb)
        total += criterion(logits.float(), yb).item() * len(yb)
        n += len(yb)
        preds.extend(logits.argmax(1).cpu().numpy())
    return total / n, np.array(preds)

def _freeze_for_scbert_finetune(model):
    """Freeze most of the PerformerLM backbone; train only the last Performer block + final norm + head.
    Mirrors TencentAILabHealthcare/scBERT/finetune.py."""
    for p in model.parameters():
        p.requires_grad = False
    backbone = model.backbone
    if hasattr(backbone, "norm"):
        for p in backbone.norm.parameters():
            p.requires_grad = True
    # scBERT's vendored performer_pytorch uses SequentialSequence(self.layers=...); some forks rename
    # to .blocks. Try the canonical name first, then fall back.
    last_block = None
    seq = getattr(getattr(backbone, "performer", None), "net", None)
    for attr in ("layers", "blocks"):
        if seq is not None and hasattr(seq, attr):
            last_block = getattr(seq, attr)[-1]
            break
    if last_block is not None:
        for p in last_block.parameters():
            p.requires_grad = True
    else:
        print("  [WARN] last Performer block not located (no .layers/.blocks on net); "
              "only head + norm will train (linear-probe mode).")
    for p in model.head.parameters():
        p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"  Trainable: {n_train:,} / {n_total:,} ({100.0*n_train/n_total:.1f}%) "
          f"— last Performer block + final norm + classification head.")


def fit_model(model, X_tr_t, y_tr, X_te_t, y_te, sample_name):
    tr_dl = DataLoader(TensorDataset(X_tr_t, torch.tensor(y_tr, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=True,
                       pin_memory=(DEVICE.type == "cuda"))
    te_dl = DataLoader(TensorDataset(X_te_t, torch.tensor(y_te, dtype=torch.long)),
                       batch_size=BATCH_SIZE, shuffle=False,
                       pin_memory=(DEVICE.type == "cuda"))

    w = 1.0 / (np.bincount(y_tr) + 1e-6)
    w = w / w.sum() * len(w)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))

    _freeze_for_scbert_finetune(model)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    ckpt = os.path.join(CKPT_DIR, f"scbert_finetune_{sample_name}_best.pt")

    best_val, pat = float("inf"), 0
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, tr_dl, optimizer, criterion)
        vl_loss, _ = eval_epoch(model, te_dl, criterion)
        scheduler.step()

        if vl_loss < best_val:
            best_val, pat = vl_loss, 0
            torch.save(model.state_dict(), ckpt)
        else:
            pat += 1

        if epoch % 5 == 0:
            print(f"    Epoch {epoch:02d} | train={tr_loss:.4f} "
                  f"val={vl_loss:.4f} | patience={pat}/{PATIENCE}")

        if pat >= PATIENCE:
            print(f"    Early stop at epoch {epoch}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    _, final_preds = eval_epoch(model, te_dl, criterion)

    runtime = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else 0
    print(f"    Done in {runtime/60:.1f} min | peak GPU: {peak_mb:.0f} MB")
    return final_preds, runtime, peak_mb

In [ ]:
def compute_metrics(y_true, y_pred, id2label):
    kappa    = cohen_kappa_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc      = matthews_corrcoef(y_true, y_pred)
    acc      = accuracy_score(y_true, y_pred)

    unique, counts = np.unique(y_true, return_counts=True)
    rarest = unique[np.argmin(counts)]
    rare_f1 = f1_score((y_true == rarest).astype(int),
                       (y_pred == rarest).astype(int), zero_division=0)
    return {
        "kappa":          round(float(kappa),    4),
        "macro_f1":       round(float(macro_f1), 4),
        "mcc":            round(float(mcc),      4),
        "accuracy":       round(float(acc),      4),
        "rare_type_f1":   round(float(rare_f1),  4),
        "rare_type_name": id2label[rarest],
        "rare_type_n":    int(counts.min()),
        "n_test":         int(len(y_true)),
        "n_classes":      int(len(id2label)),
    }

def print_metrics(m):
    print(f"  κ (kappa)    : {m['kappa']:.4f}")
    print(f"  Macro F1     : {m['macro_f1']:.4f}")
    print(f"  MCC          : {m['mcc']:.4f}")
    print(f"  Accuracy     : {m['accuracy']:.4f}")
    print(f"  Rare type F1 : {m['rare_type_f1']:.4f}  "
          f"({m['rare_type_name']}, n_test={m['rare_type_n']})")

In [ ]:
CSV_PATH = os.path.join(RESULTS_DIR, "phase8_results_scBERT.csv")

def save_result(sample_name, metrics, predictions, runtime_s, peak_gpu_mb):
    row = {
        "model":       MODEL_NAME,
        "sample":      sample_name,
        "condition":   "pretrained_finetune",
        "runtime_s":   round(runtime_s, 1),
        "peak_gpu_mb": round(peak_gpu_mb, 0),
        **metrics
    }
    row_df = pd.DataFrame([row])
    if os.path.exists(CSV_PATH):
        row_df.to_csv(CSV_PATH, mode="a", header=False, index=False)
    else:
        row_df.to_csv(CSV_PATH, mode="w", header=True,  index=False)

    pred_fname = f"{MODEL_NAME}_{sample_name}_preds.npy"
    np.save(os.path.join(RESULTS_DIR, pred_fname), predictions)
    print(f"  Saved metrics → phase8_results_scBERT.csv")
    print(f"  Saved preds   → {pred_fname}")

def check_done(sample_name):
    if not os.path.exists(CSV_PATH):
        return False
    df = pd.read_csv(CSV_PATH)
    return ((df["model"]  == MODEL_NAME)  &
            (df["sample"] == sample_name) &
            (df["condition"] == "pretrained_finetune")).any()

In [ ]:
import os
import torch
import gc

# ENSURE THE RELATIVE PATH IS RESOLVABLE
# The library looks for '../data/gene2vec_16906.npy' relative to /content/scBERT/performer_pytorch/
# Changing the working directory to the performer folder ensures '../data' maps to '/content/scBERT/data'
os.chdir('/content/scBERT/performer_pytorch')
print(f"Current Working Directory: {os.getcwd()}")

# Quick check before loop
if not os.path.exists('../data/gene2vec_16906.npy'):
    print("❌ WARNING: File still not found at '../data/gene2vec_16906.npy' relative to current path.")
else:
    print("✅ Confirmed: Library relative path is valid.")

completed, total = 0, len(SAMPLES)
print("=" * 65)
print(f"scBERT Pretrained Fine-Tuning | {total} real samples")
print("=" * 65)

for sample in SAMPLES:
    print(f"\n{'─'*65}")
    print(f"[{completed+1}/{total}]  Sample: {sample}")

    if check_done(sample):
        print(f"  [SKIP] Already in results CSV")
        completed += 1
        continue

    try:
        adata = load_sample(sample)
        train_a, test_a = stratified_split(adata)
        label2id, id2label, y_train, y_test = encode_labels(train_a, test_a)
        n_classes = len(label2id)

        gene_names = list(adata.var_names)
        X_train_dense = to_dense(train_a)
        del train_a; gc.collect()
        X_tr_t, seq_len = tokenise_for_scbert(X_train_dense, gene_names, GENE_PANEL)
        del X_train_dense; gc.collect()

        X_test_dense = to_dense(test_a)
        del adata, test_a; gc.collect()
        X_te_t, _ = tokenise_for_scbert(X_test_dense, gene_names, GENE_PANEL)
        del X_test_dense; gc.collect()

        model = scBERTClassifier(n_classes, seq_len=seq_len, pretrained_path=PRETRAINED_CKPT).to(DEVICE)

        preds, runtime, peak_mb = fit_model(model, X_tr_t, y_train, X_te_t, y_test, sample)

        metrics = compute_metrics(y_test, preds, id2label)
        print_metrics(metrics)
        save_result(sample, metrics, preds, runtime, peak_mb)

        completed += 1
        del model, X_tr_t, X_te_t
        torch.cuda.empty_cache()
        gc.collect()
    except Exception as e:
        print(f"  [ERROR] Failed processing {sample}: {e}")

os.chdir('/content') # Return to root after training
print("\n" + "=" * 65)
print(f"Complete: {completed}/{total} runs")

In [ ]:
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    mine = df[df["model"] == MODEL_NAME].copy()

    if len(mine) == 0:
        print("No results yet — run the main loop first.")
    else:
        print("=== Per-run results ===")
        cols = ["sample", "kappa", "macro_f1", "rare_type_f1", "accuracy", "runtime_s", "peak_gpu_mb"]
        print(mine[cols].sort_values("sample").to_string(index=False))

        print(f"\nTotal samples recorded: {len(mine)}/{total}")
        print(f"Total runtime         : {mine['runtime_s'].sum()/3600:.2f} hours")
        print(f"Mean κ (all samples)  : {mine['kappa'].mean():.4f}")
else:
    print(f"No CSV found at {CSV_PATH}")

In [ ]:
from google.colab import runtime
runtime.unassign()